# Capstone Project

# Forest Cover Type Prediction

By Geoffrey Arumba

## About the dataset

In the Forest Cover Type competition, we are asked to predict the forest cover type (the predominant kind of tree cover) from cartographic variables. The purpose of this EDA notebook is to provide an overview of how python visualization tools can be used to understand the complex and large dataset. EDA is the first step in this workflow where the decision-making process is initiated for the feature selection. Some valuable insights can be obtained by looking at the distribution of the target, relationship to the target and link between the features.

The study area includes four wilderness areas located in the Roosevelt National Forest of Northern Colorado. Each observation is a 30m x 30m patch. There are seven forest cover types:

1. Spruce/Fir
2. Lodgepole Pine
3. Ponderosa Pine
4. Cottonwood/Willow
5. Aspen
6. Douglas-fir
7. Krummholz

# Variables
- **Elevation** elevation in meters
- **Aspect** aspect in degrees azimuth
- **Slope** slope in degrees
- **Horizontal_Distance_To_Hydrology** Horz dist to nearest surface water features
- **Vertical_Distance_To_Hydrology** Vert dist to nearest surface water features
- **Horizontal_Distance_To_Roadways** Horz dist to nearest roadway
- **Hillshade_9am** (0 to 255 index) hillshade index at 9am, summer solstice
- **Hillshade_Noon** (0 to 255 index) hillshade index at noon, summer solstice
- **Hillshade_3pm** (0 to 255 index) hillshade index at 3pm, summer solstice
- **Horizontal_Distance_To_Fire_Points** Horz dist to nearest wildfire ignition points
- **Wilderness_Area** (4 binary cols, 0=abs or 1=pres) wilderness area designation
- **Soil_Type** (40 binary cols, 0=abs or 1=pres) soil type designation
- **Cover_Type** (7 types) forest cover type designation

## Importing important libaries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_curve, roc_auc_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Exploratory Data Analysis

### Loading the dataset into Kaggle

In [ ]:
forest_df = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')
# Displaying the head of the dataframe
forest_df.head()

### Checking for any missing values in the dataframe

In [ ]:
forest_df.info(verbose=True, show_counts=True)

There are no missing values/null values in our dataframe. Next we will drop the unique identifier **Id** as it is not one of the variables

In [ ]:
forest_df = forest_df.drop(["Id"], axis=1)
# Display the head of the dataframe
forest_df.head()

Now we are left with 55 columns

### Ckecking for the number of columns that are categorical and numerical variables

In [ ]:
categorical_variables = ["Cover_Type", "Wilderness_Area1", "Wilderness_Area2", "Wilderness_Area3", "Wilderness_Area4", "Soil_Type1", "Soil_Type2", "Soil_Type3", "Soil_Type4", "Soil_Type5", "Soil_Type6", "Soil_Type7", "Soil_Type8", "Soil_Type9", "Soil_Type10", "Soil_Type11", "Soil_Type12", "Soil_Type13", "Soil_Type14", "Soil_Type15", "Soil_Type16", "Soil_Type17", "Soil_Type18", "Soil_Type19", "Soil_Type20", "Soil_Type21", "Soil_Type22", "Soil_Type23", "Soil_Type24", "Soil_Type25", "Soil_Type26", "Soil_Type27", "Soil_Type28", "Soil_Type29", "Soil_Type30",  "Soil_Type31", "Soil_Type32", "Soil_Type33", "Soil_Type34", "Soil_Type35", "Soil_Type36", "Soil_Type37", "Soil_Type38", "Soil_Type39", "Soil_Type40"]
numerical_variables = ["Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways", "Horizontal_Distance_To_Fire_Points", "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm"]
numerical_variables_labels = ["Elevation", "Aspect", "Slope", "Horizontal Dist. To Hydrology", "Vertical Dist. To Hydrology", "Horizontal Dist. To Roadways", "Horizontal Dist. To Fire Points", "Hillshade 9am", "Hillshade Noon", "Hillshade 3pm"]

print(f'There are {len(categorical_variables)} categorical variables.')
print(f'There are {len(numerical_variables)} numerical variables.')

### Checking how the label is distributed

In [ ]:
forest_df['Cover_Type'].value_counts()

The label is balnced across the cover types

Each of the cover types sample is taken from one of four wilderness areas. Let's look at where each of the 7 cover types are found:

In [ ]:
area_1_counts = np.asarray(forest_df.groupby(["Cover_Type"]).sum()["Wilderness_Area1"])
area_2_counts = np.asarray(forest_df.groupby(["Cover_Type"]).sum()["Wilderness_Area2"])
area_3_counts = np.asarray(forest_df.groupby(["Cover_Type"]).sum()["Wilderness_Area3"])
area_4_counts = np.asarray(forest_df.groupby(["Cover_Type"]).sum()["Wilderness_Area4"])

# visualizing it in a bargraph
plt.figure(figsize=(9, 4))
p1 = plt.bar(range(1,8), area_1_counts, width=0.25, color='blue', label="Area 1")
p2 = plt.bar(range(1,8), area_2_counts, width=0.25, bottom=area_1_counts, color='yellow', label="Area 2")
p3 = plt.bar(range(1,8), area_3_counts, width=0.25, bottom=area_1_counts+area_2_counts, color='green', label="Area 3")
p4 = plt.bar(range(1,8), area_4_counts, width=0.25, bottom=area_1_counts+area_2_counts+area_3_counts, color='brown', label="Area 4")
plt.ylim([0, 2500])
plt.legend(ncol=4, loc="upper center", frameon=True, edgecolor='k')
plt.xlabel("Cover Type", fontsize=12)
plt.ylabel("Total Number", fontsize=12)
plt.title("Forest Cover Types in each Wilderness Area", fontsize=12)
plt.show()

### Distributions of numerical variables per cover type

In [ ]:
fig, axs = plt.subplots(10, 2, figsize=(14,40))

for i in range(len(numerical_variables)):
    
    sns.kdeplot(data=forest_df, x=numerical_variables[i], hue='Cover_Type', palette='turbo', fill=True, ax=axs[i][0])
    sns.boxplot(data=forest_df, y=numerical_variables[i], x='Cover_Type', palette='turbo', fliersize=2, ax=axs[i][1])
    
    axs[i][0].set_xlabel(numerical_variables_labels[i], fontsize=14)
    axs[i][1].set_ylabel(numerical_variables_labels[i], fontsize=14)
    axs[i][1].set_xlabel('Cover Type', fontsize=14)
    axs[i][0].set_ylabel('Density', fontsize=14)

plt.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.9, wspace=0.4, hspace=0.3)

plt.show()

###  Correlation heatmap for the numerical variables in relation to the cover-type

In [ ]:
plt.figure(figsize=(8,7))
corr = forest_df[numerical_variables].corr()
sns.heatmap(corr, xticklabels=numerical_variables_labels, yticklabels=numerical_variables_labels, linewidths=0.5, cmap='viridis', annot=True)
plt.show()

We can see that Elevation is quite important when it comes to classifying the  7 cover types. For example, Krummholz (class  7) is evidently found at high elevation, with practically all observations occuring at elevations above  3000m. Its elevation distribution has no overlap with the lowest lying type, which is Cottonwood/Willow (class  4), which is mainly found below  2500m.Some of numerical features seem to have little correlation with one another, e.g. Elevation and Vertical_Distance_To_Hydrology.

### Feature engineering

From our previous models fitted, we used the features as they are in the dataset. We will now consider engineering some new features by making some modifications on the existing features.

For instance we shall consider features such as **Elevation + Vertical_Distance_To_Hydrology** (which is the elevation from sea level of the nearest surface water feature from the actual observation site.

In [ ]:
forest_df['Total_Distance_To_Hydrology'] = (forest_df['Vertical_Distance_To_Hydrology']**2 + forest_df['Horizontal_Distance_To_Hydrology']**2)**0.5

forest_df['Elevation_Plus_Vertical_Hydrology'] = forest_df['Elevation'] + forest_df['Vertical_Distance_To_Hydrology']
forest_df['Elevation_Minus_Vertical_Hydrology'] = forest_df['Elevation'] - forest_df['Vertical_Distance_To_Hydrology']

forest_df['Hydrology_Plus_Fire_Points'] = forest_df['Horizontal_Distance_To_Hydrology'] + forest_df['Horizontal_Distance_To_Fire_Points']
forest_df['Hydrology_Minus_Fire_Points'] = forest_df['Horizontal_Distance_To_Hydrology'] - forest_df['Horizontal_Distance_To_Fire_Points']

forest_df['Hydrology_Plus_Roadways'] = forest_df['Horizontal_Distance_To_Hydrology'] + forest_df['Horizontal_Distance_To_Roadways']
forest_df['Hydrology_Minus_Roadways'] = forest_df['Horizontal_Distance_To_Hydrology'] - forest_df['Horizontal_Distance_To_Roadways']

forest_df['Fire_Points_Plus_Roadways'] = forest_df['Horizontal_Distance_To_Fire_Points'] + forest_df['Horizontal_Distance_To_Roadways']
forest_df['Fire_Points_Minus_Roadways'] = forest_df['Horizontal_Distance_To_Fire_Points'] - forest_df['Horizontal_Distance_To_Roadways']

Looking at our new dataframe after feature engineering

In [ ]:
forest_df.head(10)

Defining our feature and target arrays

In [ ]:
# Feature
X = forest_df.drop('Cover_Type', axis=1)
# Target
y = forest_df["Cover_Type"]
print(X.shape)
print(y.shape)

Spliting our datset into training and test dataset

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2)

Scaling our numeriacal features

In [ ]:
categorical_variables = ["Cover_Type", "Wilderness_Area1", "Wilderness_Area2", "Wilderness_Area3", "Wilderness_Area4", "Soil_Type1", "Soil_Type2", "Soil_Type3", "Soil_Type4", "Soil_Type5", "Soil_Type6", "Soil_Type7", "Soil_Type8", "Soil_Type9", "Soil_Type10", "Soil_Type11", "Soil_Type12", "Soil_Type13", "Soil_Type14", "Soil_Type15", "Soil_Type16", "Soil_Type17", "Soil_Type18", "Soil_Type19", "Soil_Type20", "Soil_Type21", "Soil_Type22", "Soil_Type23", "Soil_Type24", "Soil_Type25", "Soil_Type26", "Soil_Type27", "Soil_Type28", "Soil_Type29", "Soil_Type30",  "Soil_Type31", "Soil_Type32", "Soil_Type33", "Soil_Type34", "Soil_Type35", "Soil_Type36", "Soil_Type37", "Soil_Type38", "Soil_Type39", "Soil_Type40"]
numerical_variables = ["Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology", "Horizontal_Distance_To_Roadways", "Horizontal_Distance_To_Fire_Points", "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm", "Total_Distance_To_Hydrology", "Elevation_Plus_Vertical_Hydrology", "Elevation_Minus_Vertical_Hydrology", "Hydrology_Plus_Fire_Points", "Hydrology_Minus_Fire_Points", "Hydrology_Plus_Roadways", "Hydrology_Minus_Roadways", "Fire_Points_Plus_Roadways", "Fire_Points_Minus_Roadways"]
numerical_variables_labels = ["Elevation", "Aspect", "Slope", "Horizontal Dist. To Hydrology", "Vertical Dist. To Hydrology", "Horizontal Dist. To Roadways", "Horizontal Dist. To Fire Points", "Hillshade 9am", "Hillshade Noon", "Hillshade 3pm", "Total Dist. To Hydrology", "Elevation Plus Vertical Hydrology", "Elevation Minus Vertical Hydrology", "Hydrology Plus Fire Points", "Hydrology Minus Fire Points", "Hydrology Plus Roadways", "Hydrology Minus Roadways", "Fire Points Plus Roadways", "Fire Points Minus Roadways"]

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[numerical_variables])

X_train_scaled = X_train.copy()
X_valid_scaled = X_valid.copy()

X_train_scaled[numerical_variables] = scaler.transform(X_train_scaled[numerical_variables])
X_valid_scaled[numerical_variables] = scaler.transform(X_valid_scaled[numerical_variables])

### Model fitting

First we will define functions and algorithms that we will deploy in measuring the various metrix=cs of our models

In [ ]:
def get_metrics(model, X_test, y_test):
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')

    return accuracy, precision, recall, f1


def predict(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print('Accuracy:', accuracy_score(y_test, y_pred))
    print("F1 score (weighted average): ", f1_score(y_test, y_pred, average='weighted'))
    print("Precision (weighted average): ", precision_score(y_test, y_pred, average='weighted'))
    print("Recall (weighted average): ", recall_score(y_test, y_pred, average='weighted'))
    return y_pred


def plot_confusion_matrix(model, X_test, y_test, title):
    
    y_pred = model.predict(X_test)
    
    plt.figure(figsize=(7, 6))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap="YlGnBu")
    plt.xlabel('Predicted Class')
    plt.ylabel('Actual Class')
    plt.title(title)
    plt.xticks(ticks=np.array(range(7))+0.5, labels=['1', '2', '3', '4', '5', '6', '7'])
    plt.yticks(ticks=np.array(range(7))+0.5, labels=['1', '2', '3', '4', '5', '6', '7'])
    plt.show()
    

def plot_roc_curves(model, X_test, y_test, title):
    
    plt.figure(figsize=(8, 3.5))

    for i in range(7):

        cover = i + 1

        y_test_proba = model.predict_proba(X_test)
        auc_test = roc_auc_score(y_test==cover, y_test_proba[:,i], multi_class='multinomial')
        y_test_fpr, y_test_tpr, _ = roc_curve(y_test==cover, y_test_proba[:,i])
        
        plt.plot(y_test_fpr, y_test_tpr, marker=None, linewidth=1.5, label=f'Cover Type {cover}, AUC: {np.round(auc_test, 4)}')

        plt.ylabel('True Positive Rate (Recall)', fontsize=12)
        plt.xlabel('False Positive Rate', fontsize=12)

        plt.legend(frameon=True, edgecolor='k', fontsize=9)

        plt.title(title)

    plt.show()

### Logistic Regression Model

In [ ]:
lr_params = {'solver': ['newton-cg', 'lbfgs'],
             'penalty':['l2'],
             'C':[0.001, 0.01, 0.1, 1, 10, 100]}

lr_model = GridSearchCV(LogisticRegression(max_iter=2000, multi_class='multinomial'), param_grid=lr_params, scoring="accuracy", cv=5, verbose=1)

lr_model.fit(X_train_scaled, y_train)

In [ ]:
print("Logistic Regression best params: ", lr_model.best_params_)

print('\n')

print("Logistic Regression best score: ", lr_model.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_lr = predict(lr_model, X_valid_scaled, y_valid)

lr_model_accuracy, lr_model_precision, lr_model_recall, lr_model_f1 = get_metrics(lr_model, X_valid_scaled, y_valid)

### Decision Tree Model

In [ ]:
dt_params = {"criterion":["entropy", "gini"],
              "splitter":["best", "random"],
              "max_depth":[6, 8, 10],
              "min_samples_leaf":[1, 2, 4],
              "min_samples_split":[2, 4, 6, 8],
              "max_features":[6, 8, 10, 12],
              "class_weight":[None, "balanced"]}

dt_model = GridSearchCV(DecisionTreeClassifier(), param_grid=dt_params, scoring="accuracy", cv=5, verbose=1)

dt_model.fit(X_train_scaled, y_train)

In [ ]:
print("DT best params: ", dt_model.best_params_)

print('\n')

print("DT best score: ", dt_model.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_dtc = predict(dt_model, X_valid_scaled, y_valid)

dt_model_accuracy, dt_model_precision, dt_model_recall, dt_model_f1 = get_metrics(dt_model, X_valid_scaled, y_valid)

### Random forest model

In [ ]:
rf_params = {"n_estimators":[150],
              "max_depth":[6, 8, 10],
              "min_samples_leaf":[1, 2, 4],
              "min_samples_split":[2, 4, 6, 8],
              "max_features":[6, 8, 10, 12]}

rf_model = GridSearchCV(RandomForestClassifier(), param_grid=rf_params, scoring="accuracy", cv=5, verbose=1)

rf_model.fit(X_train_scaled, y_train)

In [ ]:
print("RF best params: ", rf_model.best_params_)

print('\n')

print("RF best score: ", rf_model.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_rfc = predict(rf_model, X_valid_scaled, y_valid)

rf_model_accuracy, rf_model_precision, rf_model_recall, rf_model_f1 = get_metrics(rf_model, X_valid_scaled, y_valid)

### K- Nearest Neighbors

In [ ]:
knn_params = {"n_neighbors":list(range(5, 20)),
              "leaf_size":[10, 20, 30, 40],
              "weights":['uniform', 'distance']}

knn_model = GridSearchCV(KNeighborsClassifier(), param_grid=knn_params, scoring="accuracy", cv=5, verbose=1)

knn_model.fit(X_train_scaled, y_train)

In [ ]:
print("KNN best params: ", knn_model.best_params_)

print('\n')

print("KNN best score: ", knn_model.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_knn = predict(knn_model, X_valid_scaled, y_valid)

knn_model_accuracy, knn_model_precision, knn_model_recall, knn_model_f1 = get_metrics(knn_model, X_valid_scaled, y_valid)

### Support Vector Machine Classifier

In [ ]:
svc_params = {'kernel':['rbf'],
               'C':[1, 5, 10, 50, 100],
               'gamma':[0.01, 0.1, 'scale', 'auto']}

svc_model = GridSearchCV(SVC(), param_grid=svc_params, scoring="accuracy", cv=5, verbose=1)

svc_model.fit(X_train_scaled, y_train)

In [ ]:
print("SVC best params: ", svc_model.best_params_)

print('\n')

print("SVC best score: ", svc_model.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_svc = predict(svc_model, X_valid_scaled, y_valid)

svc_model_accuracy, svc_model_precision, svc_model_recall, svc_model_f1 = get_metrics(svc_model, X_valid_scaled, y_valid)

### XGBoost Model

In [ ]:
xgb_params = {'max_depth':[4, 6, 8, 10],
              'n_estimators': [50, 100, 200, 300],
              'learning_rate': [0.1, 0.2, 0.3]}

xgb_model = GridSearchCV(XGBClassifier(), param_grid=xgb_params, scoring = 'accuracy', cv = 5, verbose=1)

xgb_model.fit(X_train_scaled, (y_train-1))

In [ ]:
print("XGB best params: ", xgb_model.best_params_)

print('\n')

print("XGB best score: ", xgb_model.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_xgb = predict(xgb_model, X_valid_scaled, (y_valid-1))

xgb_model_accuracy, xgb_model_precision, xgb_model_recall, xgb_model_f1 = get_metrics(xgb_model, X_valid_scaled, (y_valid-1))

### LightGBM Model

In [ ]:
lgm_params = {'n_estimators':[100, 200, 300],
              'max_depth':[4, 6, 8, 10],
              'num_leaves':[16, 32, 48, 64]}

lgm_model = GridSearchCV(LGBMClassifier(), param_grid=lgm_params, scoring="accuracy", cv=5, verbose=1)

lgm_model.fit(X_train_scaled, y_train)

In [ ]:
print("LightGBM best params: ", lgm_model.best_params_)

print('\n')

print("LightGBM best score: ", lgm_model.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_lgm = predict(lgm_model, X_valid_scaled, (y_valid))

lgm_model_accuracy, lgm_model_precision, lgm_model_recall, lgm_model_f1 = get_metrics(lgm_model, X_valid_scaled, (y_valid))

### Extra Tree Model

In [ ]:
et_params = {"n_estimators":[100, 150, 200],
              "criterion":["entropy", "gini"],
              "max_depth":[8, 12, 16, 24, None],
              "min_samples_leaf":[1, 2, 4],}

et_model_1 = GridSearchCV(ExtraTreesClassifier(), param_grid=et_params, scoring="accuracy", cv=5, verbose=1)
et_model_1.fit(X_train_scaled, y_train)

et_model_2 = GridSearchCV(ExtraTreesClassifier(), param_grid=et_params, scoring="accuracy", cv=5, verbose=1)
et_model_2.fit(X_train, y_train)

In [ ]:
print("ET_1 best params: ", et_model_1.best_params_)

print('\n')

print("ET_1 best score: ", et_model_1.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_et_1 = predict(et_model_1, X_valid_scaled, y_valid)

et_model_1_accuracy, et_model_1_precision, et_model_1_recall, et_model_1_f1 = get_metrics(et_model_1, X_valid_scaled, y_valid)

In [ ]:
print("ET_2 best params: ", et_model_2.best_params_)

print('\n')

print("ET_2 best score: ", et_model_2.best_score_)

print('\n')

print("Validation set performance: \n")
y_valid_pred_et_2 = predict(et_model_2, X_valid, y_valid)

et_model_2_accuracy, et_model_2_precision, et_model_2_recall, et_model_2_f1 = get_metrics(et_model_2, X_valid, y_valid)

### Printing the summary statistics of all our models to determine the optimal model

In [ ]:
summary_data = {'Classifier':['Logistic Regression','K Nearest Neighbors', 'Decision Tree', 'Random Forest', 'Support Vector Classifier', 'XGBoost', 'LightGBM', 'ExtraTreesModel 1', 'ExtraTreesModel 2'], 
                'Validation Set Accuracy':[lr_model_accuracy, knn_model_accuracy, svc_model_accuracy, dt_model_accuracy, rf_model_accuracy, xgb_model_accuracy, lgm_model_accuracy, et_model_1_accuracy, et_model_2_accuracy],
                'Validation Set Precision':[lr_model_precision, knn_model_precision, svc_model_precision, dt_model_precision, rf_model_precision, xgb_model_precision, lgm_model_precision, et_model_1_precision, et_model_2_precision],
                'Validation Set Recall':[lr_model_recall, knn_model_recall, svc_model_recall, dt_model_recall, rf_model_recall, xgb_model_recall, lgm_model_recall, et_model_1_recall, et_model_2_recall],
                'Validation Set F1 score':[lr_model_f1, knn_model_f1, svc_model_f1, dt_model_f1, rf_model_f1, xgb_model_f1, lgm_model_f1, et_model_1_f1, et_model_2_f1]}
svc_model_precision, 
summary_table = pd.DataFrame.from_dict(summary_data)
summary_table

#### Selecting the optimal model

From the 9 models fitted the Extra Tree model with 200 estimators and one minimum sample per leaf is the model with the highest accuracy. The accuracy, precision, recall and the F1 score are over 90%.Therefore I will choose this as the optimal model. The Logistic regression model is the worst at predicting the forest cover type.

### Generating predictions based on the optimal model on the test dataset

In [ ]:
scaler = StandardScaler()
scaler.fit(X[numerical_variables])

X_scaled = X.copy()
X_scaled[numerical_variables] = scaler.transform(X_scaled[numerical_variables])

et_best_model_1 = ExtraTreesClassifier(n_estimators=200, criterion='entropy', max_depth=None, min_samples_leaf=1)
et_best_model_1.fit(X_scaled, y)

### Feature/Variable Importance and Submission

In [ ]:
forest_df_test = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

forest_df_test = forest_df_test.drop(["Id"], axis=1)

forest_df_test['Total_Distance_To_Hydrology'] = (forest_df_test['Vertical_Distance_To_Hydrology']**2 + forest_df_test['Horizontal_Distance_To_Hydrology']**2)**0.5

forest_df_test['Elevation_Plus_Vertical_Hydrology'] = forest_df_test['Elevation'] + forest_df_test['Vertical_Distance_To_Hydrology']
forest_df_test['Elevation_Minus_Vertical_Hydrology'] = forest_df_test['Elevation'] - forest_df_test['Vertical_Distance_To_Hydrology']

forest_df_test['Hydrology_Plus_Fire_Points'] = forest_df_test['Horizontal_Distance_To_Hydrology'] + forest_df_test['Horizontal_Distance_To_Fire_Points']
forest_df_test['Hydrology_Minus_Fire_Points'] = forest_df_test['Horizontal_Distance_To_Hydrology'] - forest_df_test['Horizontal_Distance_To_Fire_Points']

forest_df_test['Hydrology_Plus_Roadways'] = forest_df_test['Horizontal_Distance_To_Hydrology'] + forest_df_test['Horizontal_Distance_To_Roadways']
forest_df_test['Hydrology_Minus_Roadways'] = forest_df_test['Horizontal_Distance_To_Hydrology'] - forest_df_test['Horizontal_Distance_To_Roadways']

forest_df_test['Fire_Points_Plus_Roadways'] = forest_df_test['Horizontal_Distance_To_Fire_Points'] + forest_df_test['Horizontal_Distance_To_Roadways']
forest_df_test['Fire_Points_Minus_Roadways'] = forest_df_test['Horizontal_Distance_To_Fire_Points'] - forest_df_test['Horizontal_Distance_To_Roadways']

X_test = forest_df_test
X_test_scaled = X_test.copy()
X_test_scaled[numerical_variables] = scaler.transform(X_test_scaled[numerical_variables])

y_test_predictions_et_best_model_1 = et_best_model_1.predict(X_test_scaled)

feat_importances_pd = pd.DataFrame(et_best_model_1.feature_importances_, index=X_test.columns, columns=['Importance'])
feat_importances_pd.sort_values(by='Importance', ascending=False)[:25]

In [ ]:
Id = np.asarray(pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')['Id'])
predictions_et_best_model_1 = pd.DataFrame({'Id':Id, 'Cover_Type':y_test_predictions_et_best_model_1})
predictions_et_best_model_1.to_csv('et_best_model_1_predictions.csv', header=True, index=False)

In [ ]:
et_best_model_2 = ExtraTreesClassifier(n_estimators=200, criterion='entropy', max_depth=None, min_samples_leaf=1)
et_best_model_2.fit(X, y)

In [ ]:
forest_df_test = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

forest_df_test = forest_df_test.drop(["Id"], axis=1)

forest_df_test['Total_Distance_To_Hydrology'] = (forest_df_test['Vertical_Distance_To_Hydrology']**2 + forest_df_test['Horizontal_Distance_To_Hydrology']**2)**0.5

forest_df_test['Elevation_Plus_Vertical_Hydrology'] = forest_df_test['Elevation'] + forest_df_test['Vertical_Distance_To_Hydrology']
forest_df_test['Elevation_Minus_Vertical_Hydrology'] = forest_df_test['Elevation'] - forest_df_test['Vertical_Distance_To_Hydrology']

forest_df_test['Hydrology_Plus_Fire_Points'] = forest_df_test['Horizontal_Distance_To_Hydrology'] + forest_df_test['Horizontal_Distance_To_Fire_Points']
forest_df_test['Hydrology_Minus_Fire_Points'] = forest_df_test['Horizontal_Distance_To_Hydrology'] - forest_df_test['Horizontal_Distance_To_Fire_Points']

forest_df_test['Hydrology_Plus_Roadways'] = forest_df_test['Horizontal_Distance_To_Hydrology'] + forest_df_test['Horizontal_Distance_To_Roadways']
forest_df_test['Hydrology_Minus_Roadways'] = forest_df_test['Horizontal_Distance_To_Hydrology'] - forest_df_test['Horizontal_Distance_To_Roadways']

forest_df_test['Fire_Points_Plus_Roadways'] = forest_df_test['Horizontal_Distance_To_Fire_Points'] + forest_df_test['Horizontal_Distance_To_Roadways']
forest_df_test['Fire_Points_Minus_Roadways'] = forest_df_test['Horizontal_Distance_To_Fire_Points'] - forest_df_test['Horizontal_Distance_To_Roadways']

X_test = forest_df_test

y_test_predictions_et_best_model_2 = et_best_model_2.predict(X_test)

feat_importances_pd = pd.DataFrame(et_best_model_2.feature_importances_, index=X_test.columns, columns=['Importance'])
feat_importances_pd.sort_values(by='Importance', ascending=False)[:25]

In [ ]:
Id = np.asarray(pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')['Id'])
predictions_et_best_model_2 = pd.DataFrame({'Id':Id, 'Cover_Type':y_test_predictions_et_best_model_2})
predictions_et_best_model_2.to_csv('et_best_model_2_predictions.csv', header=True, index=False)

From the feature importances forthe optimal model above. Apart from Elevation, which from the exploratory data analysis we predicted to be a significant predictor of cover type, it is evident that some of engineered features such as Elevation_Minus_Vertical_Hydrology and Wilderness_Area4, Elevation_Plus_Vertical_Hydrology are also very useful in predicting the forest_cover type. It is worth noting that of all the wilderness areas, only Wilderness_Area4 is a significant predictor. Individual soil types except for Soil_Type10, in contrast, did not end up holding much importance in predicting the forest cover type.